# Step 1: Research & Data Source Discovery

## 1.1 Environment Setup and Dependencies

We fetch the data from OpenStreetMap. We use the original OSM ID (osmid) as our primary identifier and calculate the exact center point (latitude and longitude) for each location.

* **Primary Source: OpenStreetMap (OSM)**: Used to extract the spatial location of employment agencies.


## 1.2 Data and Boundary Configuration

The project focuses exclusively on data within the **Berlin, Germany** boundary.

* **Spatial Integrity Plan**: Data will be joined to the **Local Reference System (LOR) boundaries** to derive the mandatory `district_id` and `neighborhood_id` for final database compliance.

In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import os 
import json

# --- 1.1 CONFIGURATION ---
# Using the specific paths and tags from your previous workflow
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}

# Update LOR_PATH to match the exact filename of the GeoJSON you uploaded
#LOR_PATH = "lor_ortsteile (1).geojson" 
#OUTPUT_PATH = "output/jobcenters_berlin.csv"

print("Libraries loaded.")
print(f"Configuration set for {PLACE_NAME} with OSM tags: {OSM_TAGS}")

# --- 1.2 LIVE DATA EXTRACTION (OSM) ---
print("\nFetching live data from OpenStreetMap (Overpass API)...")
try:
    # Fetch data and ensure the coordinate system is standard WGS84 (EPSG:4326)
    jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
    jobcenter_data_raw = gpd.GeoDataFrame(
        jobcenter_data_raw,
        geometry="geometry",
        crs="EPSG:4326"
    )
    print(f"Success! Retrieved {len(jobcenter_data_raw)} features.")
except Exception as e:
    raise RuntimeError(f"OSM extraction failed: {e}")

# --- 1.3 MANDATORY DATA CLEANING ---
# We explicitly check and report on null values in mandatory columns
print("\n--- Diagnostic Check: Nulls in Critical Columns ---")
null_counts = jobcenter_data_raw[['name', 'geometry']].isnull().sum()
print("Missing values in critical columns:")
print(null_counts)

# Drop rows missing 'name' or 'geometry' to enforce database NOT NULL compliance
initial_count = len(jobcenter_data_raw)
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()

dropped_count = initial_count - len(jobcenter_enriched)
print(f"Mandatory Drop: Removed {dropped_count} rows due to missing name/geometry.")

# --- 1.4 COORDINATE PREPARATION ---
# Extract centroids to handle both 'Point' and 'Polygon' features safely
jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x

print("\n--- Step 1 Complete ---")
print(jobcenter_enriched[['name', 'latitude', 'longitude']].head())

Libraries loaded.
Configuration set for Berlin, Germany with OSM tags: {'office': 'employment_agency'}

Fetching live data from OpenStreetMap (Overpass API)...
Success! Retrieved 65 features.

--- Diagnostic Check: Nulls in Critical Columns ---
Missing values in critical columns:
name        2
geometry    0
dtype: int64
Mandatory Drop: Removed 2 rows due to missing name/geometry.

--- Step 1 Complete ---
                                               name   latitude  longitude
element id                                                               
node    275368512   Jobcenter Mitte am Leopoldplatz  52.546772  13.356516
        1211913324           Arbeitsagentur Spandau  52.533775  13.186554
        1340158173        Jobcenter Berlin Neukölln  52.478975  13.427887
        1450906609               Agentur für Arbeit  52.578452  13.308718
        2277566662               Agentur für Arbeit  52.456592  13.411478


/tmp/ipykernel_14490/3022462177.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
/tmp/ipykernel_14490/3022462177.py:50: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x


## Step 2: Cleanup & Removing Redundancy
Explanation: Here we drop city and country because they are redundant for a Berlin project. We also remove contact:website and operator:type to keep the schema lean.
Why drop contact "website"

Maintenance: External URLs like websites change frequently. If you include them in the primary table now, the data becomes "stale" very quickly.

Scope: The current goal is to map the job centers to the Berlin District LOR system. Extra information like websites or phone numbers can be added in a later "enrichment" task once the primary table structure is approved.

Additionally: The operator:type column is a classification tag in OpenStreetMap. It tells the database who runs the facility. In the context of Berlin Job Centers, this usually indicates public. 
The center is a government-run entity (e.g., the Bundesagentur für Arbeit or local municipal government). Most Job Centers fall into this category.

In [2]:
# 1. Identify redundant columns to drop
cols_to_drop = ['addr:city', 'addr:country', 'wikidata', 'operator:type', 'contact:website']

# FIX: Changed 'gdf_raw' to 'jobcenter_data_raw' to match your notebook
jobcenter_clean = jobcenter_data_raw.drop(columns=[c for c in cols_to_drop if c in jobcenter_data_raw.columns])

# 2. Rename 'name' to 'center_name' for SQL standards and remove empty rows
jobcenter_clean = jobcenter_clean.dropna(subset=['name']).copy()
jobcenter_clean = jobcenter_clean.rename(columns={'name': 'center_name'})

# --- Verify Step 2 ---
print("--- Check Column Names ---")
print(jobcenter_clean.columns.tolist())
print("\n--- Rows remaining after cleaning ---")
print(len(jobcenter_clean))

--- Check Column Names ---
['geometry', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours', 'contact:phone', 'center_name', 'office', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date', 'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'official_name', 'smoking', 'source', 'contact:email', 'contact:fax', 'description', 'short_name', 'level', 'note', 'addr:floor', 'building:levels', 'image', 'building', 'building:colour', 'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place', 'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria', 'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name', 'name:de', 'type', 'government']

--- Rows remaining after cleaning ---
63


## Step 3: Spatial Mapping (District Join)

Explanation: Load the official Berlin district file and  use a Spatial Join to see which district polygon each job center point "falls into." This gives us the neighborhood and district names automatically.

In [3]:
LOR_PATH = "lor_ortsteile.geojson"
lor_gdf = gpd.read_file(LOR_PATH).to_crs(epsg=4326)

In [4]:
import os
print("LOR file exists:", os.path.exists(LOR_PATH))

LOR file exists: True


In [5]:
import geopandas as gpd

# 1. Rename columns based on the 'lor_ortsteile' properties found in the file
lor_gdf = lor_gdf.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# 2. Spatial Join: Mapping Job Center points to District polygons
# This uses the cleaned 'jobcenter_clean' data from your previous cell
jobcenter_mapped = gpd.sjoin(
    jobcenter_clean.reset_index(drop=True), 
    lor_gdf[['district', 'neighborhood', 'neighborhood_id', 'geometry']], 
    how='left', 
    predicate='within'
)

# --- Verification ---
print("--- Check Mapping Results ---")
print(jobcenter_mapped['district'].value_counts())

print("\n--- Check Mapped Data Preview ---")
print(jobcenter_mapped[['center_name', 'district', 'neighborhood']].head())

--- Check Mapping Results ---
district
Mitte                         14
Charlottenburg-Wilmersdorf    10
Friedrichshain-Kreuzberg       9
Neukölln                       7
Pankow                         4
Spandau                        4
Tempelhof-Schöneberg           4
Steglitz-Zehlendorf            4
Marzahn-Hellersdorf            3
Treptow-Köpenick               2
Reinickendorf                  1
Lichtenberg                    1
Name: count, dtype: int64

--- Check Mapped Data Preview ---
                       center_name              district neighborhood
0  Jobcenter Mitte am Leopoldplatz                 Mitte      Wedding
1           Arbeitsagentur Spandau               Spandau      Spandau
2        Jobcenter Berlin Neukölln              Neukölln     Neukölln
3               Agentur für Arbeit         Reinickendorf  Borsigwalde
4               Agentur für Arbeit  Tempelhof-Schöneberg    Tempelhof


## 4: Stable ID Generation and District Mapping
Deterministic Stable ID: A persistent, numeric-only ID is generated using hashlib.sha256. By hashing the geographic centroid, we ensure IDs are unique and immutable, avoiding previous AttributeError issues with different geometry types.

Official District Mapping: To comply with the final data pool schema, we map administrative district names to their official 8-digit numeric IDs (e.g., Mitte = 11001001). This ensures the data is ready for SQL relational joins.hment:** The `enrich_data_from_wikidata` function is applied to fill the `operator_name` and `contact_website` columns.

In [6]:
import hashlib

# --- 4.1 DEFINITIONS (Write once) ---
def generate_stable_id(name, lat, lon):
    """Generates a unique 10-digit ID based on name and coordinates."""
    input_data = f"{name}_{lat}_{lon}".encode('utf-8')
    hash_hex = hashlib.sha256(input_data).hexdigest()
    return int(hash_hex, 16) % (10**10)

district_mapping = {
    'Mitte': '11001001', 'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003', 'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005', 'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007', 'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009', 'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011', 'Reinickendorf': '11012012'
}

# --- 4.2 EXECUTION (The Calls) ---
# 1. Coordinate Prep (Ensuring columns exist)
jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x

# 2. Call the Stable ID function
print("Generating stable IDs...")
jobcenter_mapped['id'] = jobcenter_mapped.apply(
    lambda row: generate_stable_id(row['center_name'], row['latitude'], row['longitude']), 
    axis=1
)

# 3. Call the District mapping
print("Mapping districts...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping)

print("Step 4 complete. Data is enriched and identified.")
print("Mapping district names to official IDs...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping).astype(str)

# --- Verification ---
print("\n--- Step 4 Verification ---")
print(jobcenter_mapped[['id', 'center_name', 'district', 'district_id']].head())

Generating stable IDs...
Mapping districts...
Step 4 complete. Data is enriched and identified.
Mapping district names to official IDs...

--- Step 4 Verification ---
           id                      center_name              district  \
0  6660665090  Jobcenter Mitte am Leopoldplatz                 Mitte   
1  1092468394           Arbeitsagentur Spandau               Spandau   
2   730832232        Jobcenter Berlin Neukölln              Neukölln   
3   246338546               Agentur für Arbeit         Reinickendorf   
4  6239357044               Agentur für Arbeit  Tempelhof-Schöneberg   

  district_id  
0    11001001  
1    11005005  
2    11008008  
3    11012012  
4    11007007  


/tmp/ipykernel_14490/1859106384.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
/tmp/ipykernel_14490/1859106384.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x


## 5: Data Standardization and Final Export
Schema Compliance: The final dataset is filtered to include only the mandatory 8 columns required for the database pool: id, district_id, center_name, latitude, longitude, neighborhood, district, and neighborhood_id.

WKT & Coordinate Prep: Coordinates are extracted from the geometric centroids and formatted as numeric floats, ensuring compatibility with standard SQL spatial types.

Stable ID Integration: The deterministic IDs generated in Step 4 are finalized as the primary keys for this dataset.

Data Source Attribution: A data_source tag (OSM_LOR) is appended to ensure traceability for future audits.

In [13]:
import os

# --- 5.1 SPATIAL DATA PREP ---
# Safety check: Remove any rows with missing shapes to prevent WKT errors
jobcenter_mapped = jobcenter_mapped.dropna(subset=['geometry']).copy()

# Generate the geometry column in WKT format (e.g., POINT (13.4 52.5))
jobcenter_mapped['geometry_wkt'] = jobcenter_mapped['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# --- 5.2 FINAL SCHEMA SELECTION ---
# We MUST add 'geometry_wkt' to this list so it is included in the CSV
final_columns = [
    'id', 
    'district_id', 
    'center_name', 
    'latitude', 
    'longitude', 
    'geometry_wkt',  # <--- This was missing in your previous version
    'neighborhood', 
    'district', 
    'neighborhood_id'
]

# Create the final dataframe and add the source tag
df_final = jobcenter_mapped[final_columns].copy()
df_final['data_source'] = 'OSM_LOR'

# --- 5.3 VERIFICATION & EXPORT ---
print("--- Final Data Audit ---")
print(f"Total Records: {len(df_final)}")
print(f"Columns to Export: {df_final.columns.tolist()}")

# Preview the head to make sure 'geometry_wkt' is there
print("\n--- Data Preview ---")
print(df_final[['center_name', 'geometry_wkt']].head())

# Export to CSV
os.makedirs("output", exist_ok=True)
output_path = "output/jobcenters_berlin_final.csv"
df_final.to_csv(output_path, index=False)

print(f"\n Final file with geometry saved to {output_path}")

--- Final Data Audit ---
Total Records: 63
Columns to Export: ['id', 'district_id', 'center_name', 'latitude', 'longitude', 'geometry_wkt', 'neighborhood', 'district', 'neighborhood_id', 'data_source']

--- Data Preview ---
                       center_name                   geometry_wkt
0  Jobcenter Mitte am Leopoldplatz  POINT (13.3565162 52.5467722)
1           Arbeitsagentur Spandau  POINT (13.1865537 52.5337752)
2        Jobcenter Berlin Neukölln  POINT (13.4278868 52.4789752)
3               Agentur für Arbeit  POINT (13.3087179 52.5784523)
4               Agentur für Arbeit  POINT (13.4114781 52.4565923)

 Final file with geometry saved to output/jobcenters_berlin_final.csv
